# MVP: ProsocialDialog Safety Labeling (DSPy + TF-IDF/XGBoost)

This notebook builds two MVP workflows:
1. **DSPy + Phi-3 (Ollama via OpenAI client)** on a **stratified 50-sample subset** from `test.json` split into 25 train / 25 test.
2. **TF-IDF + XGBoost** trained on **full train + valid** and evaluated on full `test.json` and the same 25-sample DSPy test subset.

All outputs are saved under `data/results/`.

In [1]:
# If needed, run once to install dependencies in your notebook environment.
%pip install -q pandas scikit-learn xgboost dspy-ai openai

Note: you may need to restart the kernel to use updated packages.


In [1]:
from __future__ import annotations

import json
from pathlib import Path
from time import perf_counter
from typing import Any

import numpy as np
import pandas as pd
import dspy

from openai import OpenAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)

ROOT = Path.cwd()
DATA_DIR = ROOT / "data" / "prosocial-dialog"
RESULTS_DIR = ROOT / "data" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.json"
VALID_PATH = DATA_DIR / "valid.json"
TEST_PATH = DATA_DIR / "test.json"

EXPECTED_LABEL_ORDER = [
    "casual",
    "possibly_needs_caution",
    "probably_needs_caution",
    "needs_caution",
    "needs_intervention",
]

OLLAMA_MODEL = "phi3:3.8b"
OPENAI_BASE_URL = "http://localhost:11434/v1"
OPENAI_API_KEY = "ollama"  # required placeholder for Ollama OpenAI-compatible API

In [2]:
def now_stamp() -> str:
    return pd.Timestamp.utcnow().strftime("%Y%m%d_%H%M%S")


def normalize_label(value: Any) -> str:
    s = str(value).strip().lower()
    if s.startswith("__") and s.endswith("__"):
        s = s[2:-2]
    return s


def load_split_jsonl(path: Path) -> pd.DataFrame:
    df = pd.read_json(path, lines=True)
    if "context" not in df.columns or "safety_label" not in df.columns:
        raise ValueError(f"Missing required columns in {path.name}")

    df = df.copy()
    df["context"] = df["context"].fillna("").astype(str).str.strip()
    df["safety_label"] = df["safety_label"].map(normalize_label)
    df = df[df["context"] != ""].reset_index(drop=True)

    keep_cols = ["context", "response", "safety_label", "dialogue_id", "response_id", "source"]
    existing_cols = [c for c in keep_cols if c in df.columns]
    return df[existing_cols]


def evaluate_predictions(y_true: pd.Series, y_pred: pd.Series, labels: list[str]) -> dict[str, Any]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "report": classification_report(
            y_true,
            y_pred,
            labels=labels,
            zero_division=0,
            output_dict=True,
        ),
    }


def save_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=True), encoding="utf-8")

In [11]:
t = load_split_jsonl(TRAIN_PATH)
t.head()

,context,response,safety_label,dialogue_id,response_id,source
0,I tried a bit of shadowboxing today. Got one g...,"This isn't a joke about black people, right? I...",needs_caution,0,0,sbic
1,"You're right, it isn't funny. Finding enjoymen...",I am glad that you agree. Joking about abusing...,casual,0,1,sbic
2,Did I tell you how I used to poke holes in con...,That's a pretty low down thing to do.,needs_intervention,1,0,socialchemistry
3,I wanted to prank others like that,You should not poke holes in condoms. You coul...,needs_intervention,1,1,socialchemistry
4,It's harmless. I doesn't hurt anyone. I just w...,You're not thinking. How do you conclude expos...,needs_intervention,1,2,socialchemistry


In [12]:
# Load all splits according to the dataset card schema in README (JSONL records).
train_df = load_split_jsonl(TRAIN_PATH)
valid_df = load_split_jsonl(VALID_PATH)
test_df = load_split_jsonl(TEST_PATH)

all_labels = sorted(set(train_df["safety_label"]) | set(valid_df["safety_label"]) | set(test_df["safety_label"]))
label_order = [lbl for lbl in EXPECTED_LABEL_ORDER if lbl in all_labels]

print("Split sizes:")
print({"train": len(train_df), "valid": len(valid_df), "test": len(test_df)})
print("\nLabels:", label_order)
print("\nTest label distribution (for stratified sampling):")
print(test_df["safety_label"].value_counts(normalize=True).sort_index().round(4))

Split sizes:
{'train': 120236, 'valid': 20416, 'test': 25029}

Labels: ['casual', 'possibly_needs_caution', 'probably_needs_caution', 'needs_caution', 'needs_intervention']

Test label distribution (for stratified sampling):
safety_label
casual                    0.1441
needs_caution             0.4296
needs_intervention        0.1696
possibly_needs_caution    0.1211
probably_needs_caution    0.1356
Name: proportion, dtype: float64


In [14]:
train_df['safety_label'].value_counts()

safety_label
needs_caution             50493
casual                    20690
probably_needs_caution    17990
possibly_needs_caution    16458
needs_intervention        14605
Name: count, dtype: int64

In [16]:
train_df['context'].str.len().describe(), train_df['response'].str.len().describe()

(count    120236.000000
 mean         69.464711
 std          34.887971
 min           2.000000
 25%          45.000000
 50%          63.000000
 75%          86.000000
 max         672.000000
 Name: context, dtype: float64,
 count    120236.000000
 mean        137.358720
 std          84.670437
 min           4.000000
 25%          60.000000
 50%         128.000000
 75%         195.000000
 max         539.000000
 Name: response, dtype: float64)

## 1) DSPy Workflow (CoT + Few-Shot Optimizer)

We sample **50** stratified examples from `test.json`, then split to **25 train** (for few-shot optimization) and **25 test** (for evaluation).

In [17]:
# Stratified 50 sample from the official test split.
sample_50_df, _ = train_test_split(
    test_df,
    train_size=50,
    random_state=SEED,
    stratify=test_df["safety_label"],
)
sample_50_df = sample_50_df.reset_index(drop=True)

# Split 50 into 25/25 while trying to preserve label balance.
try:
    dspy_train_df, dspy_test_df = train_test_split(
        sample_50_df,
        train_size=25,
        random_state=SEED,
        stratify=sample_50_df["safety_label"],
    )
except ValueError:
    # Fallback if tiny class counts prevent stratification in the 25/25 split.
    dspy_train_df, dspy_test_df = train_test_split(
        sample_50_df,
        train_size=25,
        random_state=SEED,
        stratify=None,
    )

dspy_train_df = dspy_train_df.reset_index(drop=True)
dspy_test_df = dspy_test_df.reset_index(drop=True)

split_stamp = now_stamp()
sample50_path = RESULTS_DIR / f"mvp_dspy_sample50_{split_stamp}.csv"
dspy_train25_path = RESULTS_DIR / f"mvp_dspy_train25_{split_stamp}.csv"
dspy_test25_path = RESULTS_DIR / f"mvp_dspy_test25_{split_stamp}.csv"
sample_50_df.to_csv(sample50_path, index=False)
dspy_train_df.to_csv(dspy_train25_path, index=False)
dspy_test_df.to_csv(dspy_test25_path, index=False)

print("sample_50 label counts:")
print(sample_50_df["safety_label"].value_counts().sort_index())
print("\nDSPy train(25) label counts:")
print(dspy_train_df["safety_label"].value_counts().sort_index())
print("\nDSPy test(25) label counts:")
print(dspy_test_df["safety_label"].value_counts().sort_index())
print("\nSaved split artifacts:")
print(sample50_path)
print(dspy_train25_path)
print(dspy_test25_path)

sample_50 label counts:
safety_label
casual                     7
needs_caution             22
needs_intervention         8
possibly_needs_caution     6
probably_needs_caution     7
Name: count, dtype: int64

DSPy train(25) label counts:
safety_label
casual                     3
needs_caution             11
needs_intervention         4
possibly_needs_caution     3
probably_needs_caution     4
Name: count, dtype: int64

DSPy test(25) label counts:
safety_label
casual                     4
needs_caution             11
needs_intervention         4
possibly_needs_caution     3
probably_needs_caution     3
Name: count, dtype: int64

Saved split artifacts:
d:\Coding\VSCFiles\IndependentProjects\AI\uni-judge-research\data\results\mvp_dspy_sample50_20260420_211321.csv
d:\Coding\VSCFiles\IndependentProjects\AI\uni-judge-research\data\results\mvp_dspy_train25_20260420_211321.csv
d:\Coding\VSCFiles\IndependentProjects\AI\uni-judge-research\data\results\mvp_dspy_test25_20260420_211321.csv


In [22]:
import requests
from time import sleep
from types import SimpleNamespace

# Verify the exact model tag with manual REST calls only.
OLLAMA_TAGS_URL = "http://localhost:11434/api/tags"
OLLAMA_CHAT_URL = "http://localhost:11434/api/chat"

available_models = []
dspy_ready = False
dspy_setup_error = None


def request_json(method: str, url: str, payload: dict | None = None, *, timeout: int = 30, retries: int = 3) -> dict:
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            response = requests.request(method, url, json=payload, timeout=timeout)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            if attempt < retries:
                sleep(2 * attempt)
    raise last_error


try:
    try:
        models_data = request_json("GET", OLLAMA_TAGS_URL, timeout=10, retries=2)
        available_models = [m.get("name", "") for m in models_data.get("models", []) if m.get("name")]
        print("Available models from Ollama:")
        print(available_models[:10])

        if OLLAMA_MODEL not in available_models:
            available_phi3 = [m for m in available_models if m.startswith("phi3")]
            raise RuntimeError(
                f"Required model '{OLLAMA_MODEL}' not found. "
                f"Available phi3 models: {available_phi3}"
            )
    except RuntimeError:
        raise
    except Exception as exc:
        print(f"WARNING: Could not verify /api/tags: {exc}")
        print("Proceeding with a direct manual chat health-check against the configured tag.")

    import dspy

    class ManualOllamaLM(dspy.LM):
        """Manual request-based DSPy LM adapter for local Ollama."""

        def __init__(
            self,
            model: str,
            url: str = OLLAMA_CHAT_URL,
            temperature: float = 0.0,
            max_tokens: int = 256,
            timeout: int = 60,
            retries: int = 3,
        ):
            super().__init__(model=model, model_type="chat", temperature=temperature, max_tokens=max_tokens, cache=True)
            self.url = url
            self.timeout = timeout
            self.retries = retries

        def forward(self, prompt=None, messages=None, **kwargs):
            merged_kwargs = {**self.kwargs, **kwargs}
            chat_messages = messages or [{"role": "user", "content": prompt or ""}]

            options = {}
            if merged_kwargs.get("temperature") is not None:
                options["temperature"] = merged_kwargs["temperature"]
            if merged_kwargs.get("max_tokens") is not None:
                options["num_predict"] = int(merged_kwargs["max_tokens"])
            for key in ("top_p", "top_k", "repeat_penalty", "seed", "num_ctx"):
                if merged_kwargs.get(key) is not None:
                    options[key] = merged_kwargs[key]

            payload = {
                "model": self.model,
                "messages": chat_messages,
                "stream": False,
            }
            if options:
                payload["options"] = options

            response_payload = request_json(
                "POST",
                self.url,
                payload=payload,
                timeout=self.timeout,
                retries=self.retries,
            )
            message = response_payload.get("message") or {}
            content = str(message.get("content", "")).strip()
            if not content:
                raise RuntimeError("Ollama returned an empty message content.")

            choice = SimpleNamespace(message=SimpleNamespace(content=content))
            return SimpleNamespace(
                choices=[choice],
                model=response_payload.get("model", self.model),
                usage=response_payload.get("usage", {}),
                _hidden_params={},
            )

    lm = ManualOllamaLM(model=OLLAMA_MODEL, temperature=0.0, max_tokens=256, timeout=60, retries=3)
    dspy.settings.configure(lm=lm)

    health_output = lm(prompt="Reply with the single word ok.")
    if not health_output or not str(health_output[0]).strip():
        raise RuntimeError("Manual request health-check returned an empty response.")

    print("DSPy manual-request health check:")
    print(health_output[0])
    dspy_ready = True
    print("DSPy LM configured for Ollama via manual REST requests.")
except Exception as exc:
    dspy_setup_error = str(exc)
    print("WARNING: DSPy/Ollama setup failed. DSPy run will be skipped in this execution.")
    print(f"Reason: {dspy_setup_error}")
    print("Ensure Ollama is running, then rerun this cell:")
    print("  ollama pull phi3:3.8b")
    print("  ollama list")

Available models from Ollama:
['phi3:3.8b', 'nomic-embed-text:latest', 'gemini-3-pro-preview:latest', 'gemini-3-flash-preview:latest', 'llama3.2:1b-instruct-q4_K_M', 'llama3.2:latest']
DSPy manual-request health check:
Ok.
DSPy LM configured for Ollama via manual REST requests.


In [ ]:
class SafetyLabelSignature(dspy.Signature):
    """Classify the user context into one ProsocialDialog safety label."""

    context = dspy.InputField(desc="User utterance to classify")
    label = dspy.OutputField(
        desc=(
            "One of: " + ", ".join(label_order)
        )
    )


class SafetyClassifier(dspy.Module):
    def __init__(self):
        super().__init__()
        self.cot = dspy.ChainOfThought(SafetyLabelSignature)

    def forward(self, context: str):
        return self.cot(context=context)


def dspy_metric(example, pred, trace=None):
    true_label = normalize_label(example.label)
    pred_label = normalize_label(getattr(pred, "label", ""))
    return float(true_label == pred_label)


dspy_trainset = [
    dspy.Example(context=row.context, label=row.safety_label).with_inputs("context")
    for row in dspy_train_df.itertuples(index=False)
]

if dspy_ready:
    student = SafetyClassifier()
    optimizer = dspy.BootstrapFewShot(
        metric=dspy_metric,
        max_bootstrapped_demos=4,
        max_labeled_demos=8,
    )

    t0 = perf_counter()
    optimized_program = optimizer.compile(student=student, trainset=dspy_trainset)
    dspy_opt_seconds = perf_counter() - t0
    print(f"DSPy compile done in {dspy_opt_seconds:.2f}s")
else:
    optimized_program = None
    dspy_opt_seconds = 0.0
    print("DSPy compile skipped because LM setup is not ready.")

 72%|███████▏  | 18/25 [10:05<03:55, 33.66s/it]

Bootstrapped 4 full traces after 18 examples for up to 1 rounds, amounting to 18 attempts.
DSPy compile done in 605.93s


In [24]:
def predict_with_dspy(program, contexts: pd.Series) -> list[str]:
    preds = []
    for text in contexts:
        out = program(context=text)
        preds.append(normalize_label(getattr(out, "label", "")))
    return preds


stamp = now_stamp()
dspy_pred_path = RESULTS_DIR / f"mvp_dspy_preds_25test_{stamp}.csv"
dspy_summary_path = RESULTS_DIR / f"mvp_dspy_summary_{stamp}.json"
dspy_program_path = RESULTS_DIR / f"mvp_dspy_program_{stamp}.json"

prompt_metadata = {
    "signature": "SafetyLabelSignature",
    "signature_doc": SafetyLabelSignature.__doc__,
    "input_field": "context",
    "output_field": "label",
    "reasoning": "ChainOfThought",
    "optimizer": "BootstrapFewShot",
    "few_shot_examples": dspy_train_df[["context", "safety_label"]].to_dict(orient="records"),
    "model": OLLAMA_MODEL,
    "api_base": OPENAI_BASE_URL,
}

if optimized_program is not None:
    t1 = perf_counter()
    dspy_pred_labels = predict_with_dspy(optimized_program, dspy_test_df["context"])
    dspy_infer_seconds = perf_counter() - t1

    dspy_eval = evaluate_predictions(dspy_test_df["safety_label"], pd.Series(dspy_pred_labels), label_order)

    dspy_pred_df = dspy_test_df[["context", "safety_label"]].copy()
    dspy_pred_df["pred_label"] = dspy_pred_labels
    dspy_pred_df.to_csv(dspy_pred_path, index=False)

    dspy_summary = {
        "workflow": "dspy_phi3_fewshot_cot",
        "status": "completed",
        "sample_sizes": {"sample_50": 50, "train": 25, "test": 25},
        "timing_seconds": {
            "compile": float(dspy_opt_seconds),
            "inference": float(dspy_infer_seconds),
            "total": float(dspy_opt_seconds + dspy_infer_seconds),
        },
        "metrics": dspy_eval,
        "used_prompt": prompt_metadata,
        "artifacts": {
            "predictions_csv": str(dspy_pred_path),
            "program_state_json": str(dspy_program_path),
            "sample_50_csv": str(sample50_path),
            "train_25_csv": str(dspy_train25_path),
            "test_25_csv": str(dspy_test25_path),
        },
    }
    save_json(dspy_summary_path, dspy_summary)

    # Save program state when available for reproducibility.
    try:
        if hasattr(optimized_program, "save"):
            optimized_program.save(str(dspy_program_path))
        elif hasattr(optimized_program, "dump_state"):
            save_json(dspy_program_path, optimized_program.dump_state())
        else:
            save_json(dspy_program_path, {"info": "No program serializer available in this DSPy version."})
    except Exception as exc:
        save_json(dspy_program_path, {"error": f"Could not serialize program: {exc}"})

    print("DSPy metrics:")
    print({k: v for k, v in dspy_eval.items() if k != "report"})
    print("Saved:")
    print(dspy_pred_path)
    print(dspy_summary_path)
    print(dspy_program_path)
else:
    dspy_infer_seconds = 0.0
    dspy_eval = {"accuracy": 0.0, "macro_f1": 0.0, "weighted_f1": 0.0, "report": {}}

    dspy_summary = {
        "workflow": "dspy_phi3_fewshot_cot",
        "status": "skipped",
        "reason": dspy_setup_error or "LM setup not ready",
        "sample_sizes": {"sample_50": 50, "train": 25, "test": 25},
        "timing_seconds": {
            "compile": float(dspy_opt_seconds),
            "inference": float(dspy_infer_seconds),
            "total": float(dspy_opt_seconds + dspy_infer_seconds),
        },
        "metrics": dspy_eval,
        "used_prompt": prompt_metadata,
        "artifacts": {
            "sample_50_csv": str(sample50_path),
            "train_25_csv": str(dspy_train25_path),
            "test_25_csv": str(dspy_test25_path),
        },
    }
    save_json(dspy_summary_path, dspy_summary)

    save_json(dspy_program_path, {"status": "skipped", "reason": dspy_setup_error or "LM setup not ready"})
    print("DSPy workflow skipped. Summary saved:")
    print(dspy_summary_path)
    print(dspy_program_path)

DSPy metrics:
{'accuracy': 0.32, 'macro_f1': 0.2293916609706083, 'weighted_f1': 0.2719343814080656}
Saved:
d:\Coding\VSCFiles\IndependentProjects\AI\uni-judge-research\data\results\mvp_dspy_preds_25test_20260420_213655.csv
d:\Coding\VSCFiles\IndependentProjects\AI\uni-judge-research\data\results\mvp_dspy_summary_20260420_213655.json
d:\Coding\VSCFiles\IndependentProjects\AI\uni-judge-research\data\results\mvp_dspy_program_20260420_213655.json


## 2) TF-IDF + XGBoost Workflow (Full train + valid)

Train on all rows from `train.json` + `valid.json`, then evaluate on:
- full `test.json`
- the same DSPy 25-sample test subset

In [ ]:
train_valid_df = pd.concat([train_df, valid_df], ignore_index=True)

X_train = train_valid_df["context"].astype(str)
y_train_str = train_valid_df["safety_label"].astype(str)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_str)

xgb_pipeline = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                strip_accents="unicode",
                ngram_range=(1, 2),
                min_df=2,
                max_features=80000,
            ),
        ),
        (
            "xgb",
            XGBClassifier(
                objective="multi:softprob",
                eval_metric="mlogloss",
                n_estimators=350,
                max_depth=6,
                learning_rate=0.08,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_lambda=1.0,
                tree_method="hist",
                random_state=SEED,
                n_jobs=-1,
            ),
        ),
    ]
)

t2 = perf_counter()
xgb_pipeline.fit(X_train, y_train)
xgb_fit_seconds = perf_counter() - t2

In [ ]:

# Evaluate on full test split.
X_test_full = test_df["context"].astype(str)
y_test_full_true = test_df["safety_label"].astype(str)
y_test_full_pred = label_encoder.inverse_transform(xgb_pipeline.predict(X_test_full))
xgb_eval_full = evaluate_predictions(y_test_full_true, pd.Series(y_test_full_pred), label_order)

# Evaluate on DSPy test-25 subset for direct comparability.
X_test_25 = dspy_test_df["context"].astype(str)
y_test_25_true = dspy_test_df["safety_label"].astype(str)
y_test_25_pred = label_encoder.inverse_transform(xgb_pipeline.predict(X_test_25))
xgb_eval_25 = evaluate_predictions(y_test_25_true, pd.Series(y_test_25_pred), label_order)

xgb_stamp = now_stamp()
xgb_pred_full_path = RESULTS_DIR / f"mvp_xgb_preds_fulltest_{xgb_stamp}.csv"
xgb_pred_25_path = RESULTS_DIR / f"mvp_xgb_preds_25test_{xgb_stamp}.csv"
xgb_summary_path = RESULTS_DIR / f"mvp_xgb_summary_{xgb_stamp}.json"

pd.DataFrame({
    "context": X_test_full,
    "true_label": y_test_full_true,
    "pred_label": y_test_full_pred,
}).to_csv(xgb_pred_full_path, index=False)

pd.DataFrame({
    "context": X_test_25,
    "true_label": y_test_25_true,
    "pred_label": y_test_25_pred,
}).to_csv(xgb_pred_25_path, index=False)

xgb_summary = {
    "workflow": "tfidf_xgboost",
    "train_size": int(len(train_valid_df)),
    "timing_seconds": {"fit": float(xgb_fit_seconds)},
    "metrics": {
        "full_test": xgb_eval_full,
        "dspy_test_25": xgb_eval_25,
    },
    "model_config": {
        "vectorizer": "TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=80000)",
        "classifier": "XGBClassifier(objective=multi:softprob, n_estimators=350, max_depth=6, learning_rate=0.08)",
    },
    "artifacts": {
        "predictions_full_test_csv": str(xgb_pred_full_path),
        "predictions_25_test_csv": str(xgb_pred_25_path),
    },
}
save_json(xgb_summary_path, xgb_summary)

print(f"XGBoost fit time: {xgb_fit_seconds:.2f}s")
print("XGBoost metrics on full test:")
print({k: v for k, v in xgb_eval_full.items() if k != "report"})
print("\nXGBoost metrics on DSPy 25-test:")
print({k: v for k, v in xgb_eval_25.items() if k != "report"})
print("\nSaved:")
print(xgb_pred_full_path)
print(xgb_pred_25_path)
print(xgb_summary_path)

In [25]:
# Optional compact side-by-side summary from in-memory variables or latest saved summaries.
dspy_row = {
    "workflow": "dspy_phi3_fewshot_cot",
    "test_split": "dspy_test_25",
    "accuracy": np.nan,
    "macro_f1": np.nan,
    "time_seconds": np.nan,
}

xgb_row = {
    "workflow": "tfidf_xgboost",
    "test_split": "dspy_test_25",
    "accuracy": np.nan,
    "macro_f1": np.nan,
    "time_seconds": np.nan,
}

if "dspy_eval" in globals() and "dspy_opt_seconds" in globals() and "dspy_infer_seconds" in globals():
    dspy_row["accuracy"] = dspy_eval.get("accuracy", np.nan)
    dspy_row["macro_f1"] = dspy_eval.get("macro_f1", np.nan)
    dspy_row["time_seconds"] = dspy_opt_seconds + dspy_infer_seconds
else:
    dspy_summaries = sorted(RESULTS_DIR.glob("mvp_dspy_summary_*.json"))
    if dspy_summaries:
        latest_dspy = json.loads(dspy_summaries[-1].read_text(encoding="utf-8"))
        dspy_row["accuracy"] = latest_dspy.get("metrics", {}).get("accuracy", np.nan)
        dspy_row["macro_f1"] = latest_dspy.get("metrics", {}).get("macro_f1", np.nan)
        dspy_row["time_seconds"] = latest_dspy.get("timing_seconds", {}).get("total", np.nan)

if "xgb_eval_25" in globals() and "xgb_fit_seconds" in globals():
    xgb_row["accuracy"] = xgb_eval_25.get("accuracy", np.nan)
    xgb_row["macro_f1"] = xgb_eval_25.get("macro_f1", np.nan)
    xgb_row["time_seconds"] = xgb_fit_seconds
else:
    xgb_summaries = sorted(RESULTS_DIR.glob("mvp_xgb_summary_*.json"))
    if xgb_summaries:
        latest_xgb = json.loads(xgb_summaries[-1].read_text(encoding="utf-8"))
        dspy_test25_metrics = latest_xgb.get("metrics", {}).get("dspy_test_25", {})
        xgb_row["accuracy"] = dspy_test25_metrics.get("accuracy", np.nan)
        xgb_row["macro_f1"] = dspy_test25_metrics.get("macro_f1", np.nan)
        xgb_row["time_seconds"] = latest_xgb.get("timing_seconds", {}).get("fit", np.nan)

summary_table = pd.DataFrame([dspy_row, xgb_row])
summary_table

,workflow,test_split,accuracy,macro_f1,time_seconds
0,dspy_phi3_fewshot_cot,dspy_test_25,0.32,0.229392,1078.362053
1,tfidf_xgboost,dspy_test_25,0.44,0.195699,140.325928
